<a href="https://colab.research.google.com/github/samuelaojih/Google-Colab/blob/main/CSV_to_RIS_Converter_for_Thesis_References.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================================
# CSV to RIS Converter for Thesis References
# Purpose: Convert multiple CSV files containing academic references to RIS format
# Author: Thesis Candidate
# Date: April 2026
# ============================================================================

# ============================================================================
# SECTION 1: INSTALL AND IMPORT LIBRARIES
# ============================================================================

!pip install pandas -q

import pandas as pd
import os
import re
from google.colab import files
import zipfile

print("✅ Libraries imported successfully!")

# ============================================================================
# SECTION 2: DEFINE CONVERSION FUNCTIONS
# ============================================================================

def detect_reference_type(authors, title, journal, year, doi):
    """
    Detect reference type based on available fields.
    Returns RIS type code (JOUR, BOOK, CHAP, THES, etc.)
    """
    if journal and pd.notna(journal) and journal.strip():
        return "JOUR"  # Journal article
    elif "thesis" in str(title).lower() or "dissertation" in str(title).lower():
        return "THES"  # Thesis/Dissertation
    elif "book" in str(title).lower() or "press" in str(publisher).lower():
        return "BOOK"  # Book
    else:
        return "GEN"   # Generic


def authors_to_ris_format(authors_str):
    """
    Convert author string to RIS format.
    RIS expects: LastName, FirstName and LastName, FirstName
    """
    if pd.isna(authors_str) or not authors_str:
        return ""

    # Split multiple authors by semicolon, comma, or 'and'
    authors = re.split(r'[;,]|\sand\s', str(authors_str))

    ris_authors = []
    for author in authors:
        author = author.strip()
        if not author:
            continue

        # Check if already in "Last, First" format
        if ',' in author:
            ris_authors.append(author)
        else:
            # Try to parse "First Last" format
            parts = author.split()
            if len(parts) >= 2:
                last_name = parts[-1]
                first_name = ' '.join(parts[:-1])
                ris_authors.append(f"{last_name}, {first_name}")
            else:
                ris_authors.append(author)

    return ' and '.join(ris_authors)


def csv_to_ris(csv_file_path, output_file_path):
    """
    Convert a single CSV file to RIS format.
    """
    # Read CSV file
    df = pd.read_csv(csv_file_path)

    # Map CSV columns to RIS fields
    # Adjust these column names based on your CSV structure
    column_mapping = {
        'Title': 'TI',
        'title': 'TI',
        'Takeaway': 'N1',      # Notes field for abstract/takeaway
        'Authors': 'AU',
        'authors': 'AU',
        'Author': 'AU',
        'Year': 'PY',
        'year': 'PY',
        'Citations': 'N2',      # Number of citations
        'Abstract': 'AB',
        'abstract': 'AB',
        'Study Type': 'N1',      # Notes
        'Journal': 'JO',
        'journal': 'JO',
        'Journal SJR Quartile': 'N1',
        'DOI': 'DO',
        'doi': 'DO',
        'Consensus Link': 'UR',
    }

    ris_lines = []

    for idx, row in df.iterrows():
        # Start new reference
        ris_lines.append("TY  - " + detect_reference_type(
            row.get('Authors'), row.get('Title'), row.get('Journal'),
            row.get('Year'), row.get('DOI')
        ))

        # Title
        if pd.notna(row.get('Title')):
            ris_lines.append(f"TI  - {row['Title']}")

        # Authors
        if pd.notna(row.get('Authors')):
            authors_ris = authors_to_ris_format(row['Authors'])
            ris_lines.append(f"AU  - {authors_ris}")

        # Year
        if pd.notna(row.get('Year')):
            ris_lines.append(f"PY  - {int(row['Year'])}")

        # Journal
        if pd.notna(row.get('Journal')):
            ris_lines.append(f"JO  - {row['Journal']}")

        # DOI
        if pd.notna(row.get('DOI')):
            ris_lines.append(f"DO  - {row['DOI']}")

        # Abstract (if available)
        if pd.notna(row.get('Abstract')):
            # Clean abstract text (remove extra whitespace, newlines)
            abstract = ' '.join(str(row['Abstract']).split())
            ris_lines.append(f"AB  - {abstract}")

        # Takeaway/Notes
        if pd.notna(row.get('Takeaway')):
            takeaway = ' '.join(str(row['Takeaway']).split())
            ris_lines.append(f"N1  - Takeaway: {takeaway}")

        # Citations
        if pd.notna(row.get('Citations')):
            ris_lines.append(f"N2  - Cited by: {row['Citations']}")

        # URL
        if pd.notna(row.get('Consensus Link')):
            ris_lines.append(f"UR  - {row['Consensus Link']}")

        # End reference
        ris_lines.append("ER  - ")
        ris_lines.append("")  # Empty line between references

    # Write to file
    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(ris_lines))

    print(f"✅ Converted: {csv_file_path} -> {output_file_path}")
    print(f"   References converted: {len(df)}")

    return len(df)


def create_bibtex_from_ris(ris_file_path, bib_file_path):
    """
    Optional: Convert RIS to BibTeX using a simple mapping.
    This is a basic conversion - for production, use tools like 'biblio' or 'ris2bib'.
    """
    # This is a placeholder - for full BibTeX conversion,
    # consider using the 'ris2bib' tool or pybtex library
    print(f"⚠️  BibTeX conversion requires additional libraries.")
    print(f"   To convert RIS to BibTeX, install: !pip install rispy")
    print(f"   Then run: python -c 'import rispy; ...'")

# ============================================================================
# SECTION 3: UPLOAD CSV FILES
# ============================================================================

print("=" * 70)
print("CSV TO RIS CONVERTER")
print("=" * 70)
print("\nPlease upload your CSV files containing references.")
print("Supported file names: geomorphology.csv, human_environment.csv, etc.")
print("\n" + "=" * 70)

# Upload files
uploaded = files.upload()

# Create output directory
os.makedirs('/content/ris_output', exist_ok=True)
os.makedirs('/content/csv_uploads', exist_ok=True)

# Save uploaded files
csv_files = []
for filename, content in uploaded.items():
    filepath = f"/content/csv_uploads/{filename}"
    with open(filepath, 'wb') as f:
        f.write(content)
    csv_files.append(filepath)
    print(f"✅ Uploaded: {filename}")

# ============================================================================
# SECTION 4: CONVERT EACH CSV TO RIS
# ============================================================================

print("\n" + "=" * 70)
print("CONVERTING CSV FILES TO RIS FORMAT")
print("=" * 70)

ris_files = []
total_refs = 0

for csv_file in csv_files:
    # Generate output filename
    base_name = os.path.splitext(os.path.basename(csv_file))[0]
    ris_file = f"/content/ris_output/{base_name}.ris"

    # Convert
    try:
        num_refs = csv_to_ris(csv_file, ris_file)
        ris_files.append(ris_file)
        total_refs += num_refs
    except Exception as e:
        print(f"❌ Error converting {csv_file}: {e}")

print("\n" + "=" * 70)
print(f"✅ CONVERSION COMPLETE!")
print(f"   Total references converted: {total_refs}")
print(f"   RIS files created: {len(ris_files)}")
print("=" * 70)

# ============================================================================
# SECTION 5: DISPLAY PREVIEW OF CONVERTED FILES
# ============================================================================

print("\n" + "=" * 70)
print("PREVIEW OF CONVERTED RIS FILES")
print("=" * 70)

for ris_file in ris_files[:3]:  # Show first 3 files only
    print(f"\n📄 {os.path.basename(ris_file)}:")
    print("-" * 40)
    with open(ris_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()[:20]  # Show first 20 lines
        print(''.join(lines))
    print("-" * 40)

# ============================================================================
# SECTION 6: CREATE SINGLE COMBINED RIS FILE
# ============================================================================

print("\n" + "=" * 70)
print("CREATING COMBINED RIS FILE")
print("=" * 70)

combined_ris_path = "/content/ris_output/all_references_combined.ris"

with open(combined_ris_path, 'w', encoding='utf-8') as outfile:
    for ris_file in ris_files:
        with open(ris_file, 'r', encoding='utf-8') as infile:
            outfile.write(infile.read())
            outfile.write("\n")  # Add extra newline between files

print(f"✅ Combined RIS file created: {combined_ris_path}")
print(f"   Contains all {total_refs} references from {len(ris_files)} files")

# ============================================================================
# SECTION 7: CREATE ZIP FILE FOR DOWNLOAD
# ============================================================================

print("\n" + "=" * 70)
print("CREATING ZIP FILE FOR DOWNLOAD")
print("=" * 70)

zip_path = "/content/thesis_references_ris.zip"

with zipfile.ZipFile(zip_path, 'w') as zipf:
    # Add individual RIS files
    for ris_file in ris_files:
        zipf.write(ris_file, os.path.basename(ris_file))
    # Add combined RIS file
    zipf.write(combined_ris_path, os.path.basename(combined_ris_path))
    # Also add a README file
    readme_content = """# Thesis References in RIS Format

This archive contains your references converted from CSV to RIS format.

## Files Included:
- *.ris - Individual RIS files for each CSV source
- all_references_combined.ris - All references combined into a single file

## How to Import to Reference Managers:

### Zotero:
1. Open Zotero
2. File → Import → Select RIS file
3. Choose the combined RIS file or individual files

### Mendeley:
1. Open Mendeley Desktop
2. File → Add Files → Select RIS file(s)
3. Mendeley will automatically parse the RIS format

### EndNote:
1. Open EndNote
2. File → Import → File
3. Select RIS file and choose "RIS" as import option

## Total References: {total_refs}
## Source Files: {len(ris_files)}

Generated by CSV to RIS Converter - April 2026
"""
    zipf.writestr("README.txt", readme_content)

print(f"✅ ZIP file created: {zip_path}")

# ============================================================================
# SECTION 8: DOWNLOAD THE ZIP FILE
# ============================================================================

print("\n" + "=" * 70)
print("DOWNLOADING ZIP FILE")
print("=" * 70)

files.download(zip_path)

print("\n✅ Download complete!")
print("📁 File saved as: 'thesis_references_ris.zip'")
print("\n" + "=" * 70)
print("INSTRUCTIONS FOR USING RIS FILES")
print("=" * 70)
print("""
1. Extract the ZIP file to access your RIS files
2. Import the RIS file(s) into your reference manager:
   - Zotero: File → Import → Select RIS file
   - Mendeley: File → Add Files → Select RIS file
   - EndNote: File → Import → File (choose RIS option)
3. The combined file 'all_references_combined.ris' contains all references

Note: RIS format preserves:
- Authors, Title, Year, Journal, DOI
- Abstracts and Notes
- URLs and Citation counts
""")
print("=" * 70)
print("END OF CONVERSION SCRIPT")
print("=" * 70)

✅ Libraries imported successfully!
CSV TO RIS CONVERTER

Please upload your CSV files containing references.
Supported file names: geomorphology.csv, human_environment.csv, etc.



Saving Geomorphological Theory in geograpgy (social sciences) - Apr 20, 2026.csv to Geomorphological Theory in geograpgy (social sciences) - Apr 20, 2026.csv
Saving Human-Environment Interaction Theory  in geograpgy (social sciences) - Apr 20, 2026.csv to Human-Environment Interaction Theory  in geograpgy (social sciences) - Apr 20, 2026.csv
Saving Sustainable Land Management Theory (social sciences) - Apr 20, 2026.csv to Sustainable Land Management Theory (social sciences) - Apr 20, 2026.csv
Saving system theory in geograpgy (social sciences) - Apr 20, 2026.csv to system theory in geograpgy (social sciences) - Apr 20, 2026.csv
Saving Watershed Theory - Apr 20, 2026.csv to Watershed Theory - Apr 20, 2026.csv
✅ Uploaded: Geomorphological Theory in geograpgy (social sciences) - Apr 20, 2026.csv
✅ Uploaded: Human-Environment Interaction Theory  in geograpgy (social sciences) - Apr 20, 2026.csv
✅ Uploaded: Sustainable Land Management Theory (social sciences) - Apr 20, 2026.csv
✅ Uploaded: 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download complete!
📁 File saved as: 'thesis_references_ris.zip'

INSTRUCTIONS FOR USING RIS FILES

1. Extract the ZIP file to access your RIS files
2. Import the RIS file(s) into your reference manager:
   - Zotero: File → Import → Select RIS file
   - Mendeley: File → Add Files → Select RIS file
   - EndNote: File → Import → File (choose RIS option)
3. The combined file 'all_references_combined.ris' contains all references

Note: RIS format preserves:
- Authors, Title, Year, Journal, DOI
- Abstracts and Notes
- URLs and Citation counts

END OF CONVERSION SCRIPT
